# PREREQUISITES:
download s0.tar.gz to the Exercise_13 directory!

In [ ]:
!wget https://drive.switch.ch/index.php/s/Sa2C6i9dNK27J6P/download -O s0.tar.gz
!mv s0 s0.bak && mv data_s0 data_s0.bak
!tar -xzf s0.tar.gz



# Note, the compile_2_new.bash should give you a pretty fast multicore lmp! check it out...


# Exercise 13 - 2026: Active learning with the reaction of exercise 12

## In this exercise we want to train a potential and test some active learning on the simple model. We thank Dr. Umberto Raucci for the collaboration. The reaction is similar to the one studied by Umberto Raucci in this [paper](https://pubs.acs.org/doi/10.1021/acs.jpclett.5c00688)



# Extract the tar file of s0 and data_s0 from the `Exercise 13 ` directory. You can also do that from a termina, or execute the whole cell above. At the end you need to have s0 and data_s0 in the main path of the exercise.

`tar -zxvf s0.tar.gz`

## The subdirectories of Exercise 13 contain data and active training scripts as described in the paper and in the slides of Umberto. Run the scripts of the steps of the active learning and understand the different steps. The training then could take place in the directory s0/02_iteration/1model_redo but it is too long. Then we use the existing results. 

## 1. Have a look at the model_dev.out in the 00_iteration step:

`cd s0/00_iteration/dyn_bias/b120_data`  and see the complete (3.8 ns) OPES trajectory.


In [ ]:
import matplotlib.pyplot as plt

def read_col4(path):
    values = []
    with open(path, "r") as file:
        for line in file:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    values.append(float(parts[4]))
    return values

col4_a = read_col4("s0/00_iteration/dyn_bias/b120_data/model_devi.out")

plt.figure(figsize=(8, 5))
plt.hist(col4_a, bins=200, edgecolor='black', alpha=0.5, density=True, label='Iteration 0')
plt.xlim((0, 0.6))
plt.xlabel("Force Deviation among 4 potentials")
plt.ylabel("Frequency")
plt.title("Max dev forces (eV/A/atom), iteration 0")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## Prepare a hystogram of the COLVAR of 00_iteration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def read_col(path, col_index):
    values = []
    with open(path, "r") as file:
        for line in file:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) > col_index:
                    values.append(float(parts[col_index]))
    return values

cv_values = read_col("s0/00_iteration/dyn_bias/b120_data/COLVAR", 3)  # <-- change path; CV is column index 3

plt.figure(figsize=(8, 5))
plt.hist(cv_values, bins=100, edgecolor='black', alpha=0.7)
plt.xlabel("CV")
plt.ylabel("Frequency")
plt.title("Distribution of CV")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# 2. In the directory s0/00_iteration/dyn_bias/coord_selected, have a look at `clustering_model_dev.py` and try to understand how the configurations are selected from this histogram. The command to select the active learned configurations is 

`python clustering_model_dev.py > output.active`


## Visualize the selected configurations, and do an histogram of the COLVAR of those. 

In [ ]:
import numpy as np
import scipy
from scipy.special import sph_harm
from glob import glob
from ase.io import read,write
from ase import neighborlist
import matplotlib.pyplot as plt

import numpy as np
from ase.visualize import view
import matplotlib.pyplot as plt
import nglview as nv


import ase
import ase.io
import ase.lattice.cubic

from ase.units import fs, kB
from ase.calculators.lammpsrun import LAMMPS

def view_structure(system):
    t = nv.ASEStructure(system) 
    w = nv.NGLWidget(t, gui=True)
    w.add_spacefill()
    return w


def view_trajectory(trajectory):
    t2 = nv.ASETrajectory(trajectory)
    w2 = nv.NGLWidget(t2, gui=True)
    w2.add_representation('label',label_type='atomindex',color='black')
    w2.add_representation('licorice')
    w2.add_representation('spacefill',selection=[5],color="red",radius=0.7)
    w2.add_representation('ball_and_stick')

    return w2







In [ ]:
selected = read('s0/00_iteration/dyn_bias/coord_selected/check_cv_selected/coords_selected.xyz',":")
view_trajectory(selected)

In [ ]:
#
# Careful in this case cv is the second column
#

import matplotlib.pyplot as plt
import numpy as np

def read_col(path, col_index):
    values = []
    with open(path, "r") as file:
        for line in file:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) > col_index:
                    values.append(float(parts[col_index]))
    return values

cv_values = read_col("s0/00_iteration/dyn_bias/coord_selected/check_cv_selected//COLVAR", 1)  # <-- change path; CV is column index 1

plt.figure(figsize=(8, 5))
plt.hist(cv_values, bins=40, edgecolor='black', alpha=0.7)
plt.xlabel("CV")
plt.ylabel("Frequency")
plt.title("Distribution of CV")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# 3. In the dir dft_sp you find the (already computed) energies and forces for these active learning points, in `s0/01_iteration/1model_again` you find the dir to train an example of active trained model.


# 4. "Pretend to have run" the 01 Iteration OPES, 

In [ ]:
import matplotlib.pyplot as plt

def read_col4(path):
    values = []
    with open(path, "r") as file:
        for line in file:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    values.append(float(parts[4]))
    return values

col4_a = read_col4("s0/01_iteration/dyn_bias/b120/model_devi.out")

plt.figure(figsize=(8, 5))
plt.hist(col4_a, bins=200, edgecolor='black', alpha=0.5, density=True, label='Iteration 0')
plt.xlim((0, 0.6))
plt.xlabel("Force Deviation among 4 potentials")
plt.ylabel("Frequency")
plt.title("Max dev forces (eV/A/atom), iteration 1")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend()
plt.show()

# 4b. ...and see the selection of the active learned configurations in the `s0/01_iteration/1model/dyn_bias/coord_selected` path. This time the criterion is CV + standard deviation among the 4 potentials. Run the selection... look at the outputs

# 5. Run a OPES MD using the potential 02_iterations in the directory `~/MMM_2026/Exercise_13/s0/02_iteration/dyn_bias/b120_opes` , compare the results from the three iterations 

### The important directory in the dyn_bias path is the coord_selected directory where the active learning configurations are chosen.


In [ ]:
import matplotlib.pyplot as plt

def read_col4(path):
    values = []
    with open(path, "r") as file:
        for line in file:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    values.append(float(parts[4]))
    return values

col4_a = read_col4("s0/00_iteration/dyn_bias/b120_data/model_devi.out")

col4_b = read_col4("s0/01_iteration/dyn_bias/b120/model_devi.out")
col4_c = read_col4("s0/02_iteration/dyn_bias/b120_opes/model_devi.out")  

plt.figure(figsize=(8, 5))
plt.hist(col4_a, bins=200, edgecolor='black', alpha=0.5, density=True, label='Iteration 0')
plt.hist(col4_b, bins=200, edgecolor='black', alpha=0.5, density=True, label='Iteration 1')
plt.hist(col4_c, bins=200, edgecolor='black', alpha=0.5, density=True, label='Iteration 2')

plt.xlim((0, 0.6))
plt.xlabel("Force Deviation among 4 potentials")
plt.ylabel("Frequency")
plt.title("Max dev forces (eV/A/atom), comparison between three iterations")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend()
plt.show()